# S5 Low-Noise MC Comparison

This notebook uses the dataframe-plus-dictionary pattern we discussed:

- `run_df`: one row per experiment with metadata and a stable `run_id`
- `run_data[run_id]`: eval history, train-log history, and local metadata when available

Current focus:

- offline BC MC via `sample_then_corrupt`
- NAIL-OPD forward-KL MC via `forward_kl_simple`
- low-noise regime `eta in {0.05, 0.1, 0.2}`
- plus `eta = 0.3` for NAIL-OPD

The summary numbers in the manifest come from `s5_experiment_log.md`. If you later export or sync the actual run folders and logs into this repo, the same notebook will automatically pick up local curves from `eval_history.jsonl` and the wrapper log files.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.s5_runs import build_summary_table, load_manifest, load_run_data, stack_history

run_df = load_manifest(root=PROJECT_ROOT)
run_data = load_run_data(run_df)
comparison_df = (
    run_df.loc[run_df["comparison_group"] == "low_noise_mc_focus"]
    .sort_values(["plot_group", "eta"])
    .reset_index(drop=True)
)

comparison_df[[
    "run_id",
    "plot_group",
    "eta",
    "objective",
    "summary_val_loss",
    "summary_val_clean_full_exact",
    "out_dir_resolved",
    "train_log_path_resolved",
]]


In [ ]:
build_summary_table(comparison_df)


In [ ]:
clean_exact_pivot = (
    comparison_df.pivot(
        index="eta",
        columns="plot_group",
        values="summary_val_clean_full_exact",
    )
    .sort_index()
)
clean_exact_pivot["NAIL_minus_offline"] = (
    clean_exact_pivot.get("NAIL-OPD forward-KL MC")
    - clean_exact_pivot.get("Offline BC MC")
)
clean_exact_pivot


In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook needs matplotlib for plotting. Install it in your notebook environment or switch to a plotting-ready kernel."
    ) from exc

COLORS = {
    "Offline BC MC": "#355070",
    "NAIL-OPD forward-KL MC": "#C1121F",
}
MARKERS = {
    "Offline BC MC": "o",
    "NAIL-OPD forward-KL MC": "D",
}

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "#FAF7F2",
        "axes.edgecolor": "#333333",
        "axes.grid": True,
        "grid.color": "#D8D2C8",
        "grid.alpha": 0.8,
        "grid.linestyle": "--",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "legend.frameon": False,
    }
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
plot_metrics = [
    ("summary_val_clean_full_exact", "Clean Full Exact", "Clean full exact vs noise"),
    ("summary_val_loss", "Validation Loss", "Validation loss vs noise"),
]

for ax, (metric, ylabel, title) in zip(axes, plot_metrics):
    ax.axvspan(0.045, 0.205, color="#EADFCF", alpha=0.55)
    for label, group in comparison_df.groupby("plot_group"):
        ordered = group.sort_values("eta")
        ax.plot(
            ordered["eta"],
            ordered[metric],
            color=COLORS[label],
            marker=MARKERS[label],
            linewidth=2.5,
            markersize=7,
            label=label,
        )
    ax.set_title(title)
    ax.set_xlabel("Noise level $\\eta$")
    ax.set_ylabel(ylabel)
    ax.set_xticks(sorted(comparison_df["eta"].unique()))

axes[0].set_ylim(0.88, 1.005)
axes[0].legend(loc="lower left")
fig.suptitle("Offline BC MC vs NAIL-OPD forward-KL MC", fontsize=16, y=1.03)
plt.show()


In [ ]:
eval_curves = stack_history(
    comparison_df,
    run_data,
    history_key="eval_history",
    metric="val/clean_full_exact",
)

if eval_curves.empty:
    print(
        "No local eval histories were found yet. Once the run folders with eval_history.jsonl are present, this cell will plot them automatically."
    )
else:
    fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
    for (label, eta), group in eval_curves.groupby(["plot_group", "eta"]):
        ordered = group.sort_values("iter")
        ax.plot(
            ordered["iter"],
            ordered["value"],
            color=COLORS[label],
            linewidth=2,
            alpha=0.92,
            label=f"{label}, eta={eta}",
        )
    ax.set_title("Eval curves: clean full exact")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("val/clean_full_exact")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.show()


In [ ]:
train_curves = stack_history(
    comparison_df,
    run_data,
    history_key="train_history",
    metric="loss",
)

if train_curves.empty:
    print(
        "No local train-log histories were found yet. If the wrapper logs exist at the manifest paths, this cell will pick them up automatically."
    )
else:
    fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
    for (label, eta), group in train_curves.groupby(["plot_group", "eta"]):
        ordered = group.sort_values("iter")
        ax.plot(
            ordered["iter"],
            ordered["value"],
            color=COLORS[label],
            linewidth=1.8,
            alpha=0.9,
            label=f"{label}, eta={eta}",
        )
    ax.set_title("Train curves from stdout logs")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("loss")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.show()


## Extending This

To add more runs later, append rows to `analysis/manifests/s5_low_noise_comparison.csv` with:

- a new `run_id`
- the experiment features you care about (`eta`, method, objective, law, etc.)
- an `out_dir` if you have local run folders
- a `train_log_path` if you want to parse stdout training curves
- summary metrics if you want plots to work even before local histories are present
